# NAME:Christian MANZI SHEMA
# STUDENT ID: 101382
# COURSE:ADVANCED BIG DATA
# INSTRUCTOR: Temitope Oguntade

# 00 · Environment, Data Generation & Data Profiling
**Big Data Analytics – Final Project (Oct 2026)**

**Goal of this notebook:** prepare everything the other notebooks depend on.

| Step | What we do | Why |
|---|---|---|
| 1 | Check Python / Java / Docker / library versions | Spark needs Java; Mongo + HBase run in Docker |
| 2 | Generate the synthetic dataset | The assignment's generator, made scalable |
| 3 | Verify files | Make sure every later notebook has its input |
| 4 | Profile the raw data | Find real quirks *before* designing schemas |

**Run order of the whole project:** `00 → 01 → 02 → 03 → 04 → 05 → 06` (01 and 02 need Docker; 03–06 also run without it).

## Step 1 · Check the environment

In [1]:
import sys, subprocess, shutil, platform
sys.path.append("../src")
from config import *

print("Python  :", sys.version.split()[0], "|", platform.system(), platform.machine())
print("Project :", ROOT)

# Java is required by Spark (use Java 17 or 21)
java = shutil.which("java")
print("Java    :", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0] if java else "NOT FOUND  -> install JDK 17")

# Docker is required for MongoDB + HBase (notebooks 01, 02)
docker = shutil.which("docker")
print("Docker  :", subprocess.run(["docker", "--version"], capture_output=True, text=True).stdout.strip() if docker else "NOT FOUND -> install Docker Desktop")

for lib in ["pandas", "numpy", "faker", "pymongo", "happybase", "pyspark", "matplotlib", "seaborn", "pyarrow"]:
    try:
        mod = __import__(lib); print(f"{lib:11s}:", getattr(mod, "__version__", "ok"))
    except ImportError:
        print(f"{lib:11s}: MISSING -> pip install -r requirements.txt")

Python  : 3.13.14 | Windows AMD64
Project : E:\bigdata\auca-ecommerce-analytics
Java    : java version "17.0.12" 2024-07-16 LTS
Docker  : Docker version 29.8.0, build 88096ef
pandas     : 3.1.1
numpy      : 2.5.4
faker      : ok
pymongo    : 4.19.0
happybase  : 1.3.0
pyspark    : 4.0.5
matplotlib : 3.11.2
seaborn    : 0.13.2
pyarrow    : 26.0.1


## Step 2 · Generate the dataset

The generator (`src/dataset_generator.py`) is the course script with **one change**: the size is chosen by the `SCALE` variable.

| SCALE | Users | Products | Transactions | Sessions | Approx. disk | Approx. RAM while generating | Use for |
|---|---|---|---|---|---|---|---|
| `small` | 2,000 | 1,000 | 10,000 | 40,000 | ~80 MB | < 1 GB | quick tests (≈10 s) |
| **`medium`** | 10,000 | 5,000 | 50,000 | 200,000 | ~400 MB | ~2–3 GB | **recommended – the whole project** |
| `full` | 10,000 | 5,000 | 500,000 | 2,000,000 | ~4 GB | **15+ GB** | only on a big machine |

> The original script keeps *every* session in a Python list before writing. At 2 M sessions that needs far more RAM than most laptops have – this is why `medium` is the default. The analysis logic is identical at any scale (and the report should say so in its *scalability* section).

In [2]:
import os, subprocess, sys

SCALE = os.environ.get("SCALE", "small")   # <- "small" = fast dry run, "medium" = recommended, "full" = needs 16+ GB RAM
FORCE_REGENERATE = False  # True = overwrite existing files

already = (RAW / "transactions.json").exists() and (RAW / "sessions_0.json").exists()
if already and not FORCE_REGENERATE:
    print("Dataset already exists in", RAW, "-> skipping generation (set FORCE_REGENERATE=True to redo)")
else:
    env = {**os.environ, "SCALE": SCALE, "OUT_DIR": str(RAW)}
    with timer(f"generate dataset (SCALE={SCALE})"):
        p = subprocess.Popen([sys.executable, str(ROOT / "src" / "dataset_generator.py")],
                             cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line.rstrip())
        p.wait()
    assert p.returncode == 0, "Generator failed - read the messages above"

Dataset already exists in E:\bigdata\auca-ecommerce-analytics\data\raw -> skipping generation (set FORCE_REGENERATE=True to redo)


## Step 3 · Verify the files

In [3]:
import pandas as pd
rows = [{"file": f.name, "size_MB": round(f.stat().st_size / 1e6, 1)} for f in sorted(RAW.glob("*.json"))]
files = pd.DataFrame(rows); display(files)
print("Total:", files.size_MB.sum().round(1), "MB")
assert {"users.json", "products.json", "categories.json", "transactions.json"} <= set(files.file)
assert any(files.file.str.startswith("sessions_"))

,file,size_MB
0,categories.json,0.0
1,products.json,1.6
2,sessions_0.json,187.9
3,sessions_1.json,188.2
4,transactions.json,21.9
5,users.json,1.8


Total: 401.4 MB


## Step 4 · Profile the raw data

Before designing any schema we must know what is *really* in the files. The assignment text describes the data in general terms; the generator is the ground truth.
We load everything with plain Python/pandas here (fine at this size) and write the findings to `outputs/tables/data_quality_findings.csv` – they are reused in the report's *Limitations* section.

In [4]:
import json, glob
users = json.load(open(RAW / "users.json"))
products = json.load(open(RAW / "products.json"))
categories = json.load(open(RAW / "categories.json"))
transactions = json.load(open(RAW / "transactions.json"))
sessions = []
for f in sorted(glob.glob(str(RAW / "sessions_*.json"))):
    sessions += json.load(open(f))

print(f"users={len(users):,}  products={len(products):,}  categories={len(categories):,}  "
      f"transactions={len(transactions):,}  sessions={len(sessions):,}")
print("\nUser keys       :", list(users[0]))
print("Product keys    :", list(products[0]))
print("Session keys    :", list(sessions[0]))
print("Transaction keys:", list(transactions[0]))

users=10,000  products=5,000  categories=25  transactions=50,000  sessions=200,000

User keys       : ['user_id', 'geo_data', 'registration_date', 'last_active']
Product keys    : ['product_id', 'name', 'category_id', 'base_price', 'current_stock', 'is_active', 'price_history', 'creation_date']
Session keys    : ['session_id', 'user_id', 'start_time', 'end_time', 'duration_seconds', 'geo_data', 'device_profile', 'viewed_products', 'page_views', 'cart_contents', 'conversion_status', 'referrer']
Transaction keys: ['transaction_id', 'session_id', 'user_id', 'timestamp', 'items', 'subtotal', 'discount', 'total', 'payment_method', 'status']


In [5]:
import collections as C

findings = []
def finding(topic, value, impact):
    findings.append({"topic": topic, "value": value, "impact_on_design": impact}); print(f"- {topic}: {value}")

# 1) Schema gaps versus the assignment text -----------------------------------------------------
finding("products have subcategory_id?", "subcategory_id" in products[0],
        "Assignment text shows it, generator does not create it -> margin only at CATEGORY level (avg of subcategory margins)")
finding("users have age / birth date?", any(k in users[0] for k in ("age", "birth_date", "dob")),
        "No age -> 'users by age group' impossible; segment by purchase frequency, registration cohort and geography instead")

# 2) Session / transaction consistency ----------------------------------------------------------
status = C.Counter(s["conversion_status"] for s in sessions)
conv = {s["session_id"] for s in sessions if s["conversion_status"] == "converted"}
txn_sessions = {t["session_id"] for t in transactions if t["session_id"]}
finding("conversion_status counts", dict(status), "browsed / abandoned / converted")
finding("'converted' sessions WITHOUT a transaction", f"{len(conv - txn_sessions):,} of {len(conv):,} ({len(conv - txn_sessions)/max(len(conv),1):.0%})",
        "conversion_status is NOT reliable (stock failures) -> true purchases are taken from transactions")
finding("transactions WITHOUT a session", f"{sum(t['session_id'] is None for t in transactions):,} of {len(transactions):,}",
        "Roughly half the orders cannot be joined to browsing -> funnel must treat them separately")

# 3) Arithmetic & integrity checks --------------------------------------------------------------
bad_math = sum(abs(t["subtotal"] - t["discount"] - t["total"]) > 0.011 for t in transactions)
finding("transactions where subtotal - discount != total", bad_math, "0 expected -> totals can be trusted")
finding("duplicate session ids", len(sessions) - len({s["session_id"] for s in sessions}), "0 expected")
finding("sessions with empty viewed_products", sum(not s["viewed_products"] for s in sessions), "kept; they simply never reach a product page")
finding("inactive products", sum(not p["is_active"] for p in products), "~5% by design")
finding("products with zero stock at the end", sum(p["current_stock"] == 0 for p in products), "inventory chain works")

# 4) Time coverage ------------------------------------------------------------------------------
finding("session time range", f"{min(s['start_time'] for s in sessions)[:10]} -> {max(s['start_time'] for s in sessions)[:10]}",
        "dates are relative to the day you generate the data (not fixed 2025 dates)")
finding("user registration range", f"{min(u['registration_date'] for u in users)[:10]} -> {max(u['registration_date'] for u in users)[:10]}",
        "all users registered BEFORE the activity window -> cohort analysis is by registration month, 3-9 months of tenure")

pd.DataFrame(findings).to_csv(TAB / "data_quality_findings.csv", index=False)

- products have subcategory_id?: False
- users have age / birth date?: False
- conversion_status counts: {'browsed': 69830, 'converted': 40609, 'abandoned': 89561}
- 'converted' sessions WITHOUT a transaction: 15,515 of 40,609 (38%)
- transactions WITHOUT a session: 24,906 of 50,000
- transactions where subtotal - discount != total: 0
- duplicate session ids: 0
- sessions with empty viewed_products: 2578
- inactive products: 234
- products with zero stock at the end: 163
- session time range: 2026-08-11 -> 2026-11-09
- user registration range: 2026-02-12 -> 2026-08-11


In [6]:
# Quick look at the shape of the browsing data (feeds the funnel design later)
pt = C.Counter(pv["page_type"] for s in sessions for pv in s["page_views"])
print("Page-view mix:"); display(pd.Series(pt).sort_values(ascending=False).to_frame("page_views"))
print("Sessions with a non-empty cart:", sum(bool(s["cart_contents"]) for s in sessions))
print("Sessions that visited the checkout page:", sum(any(pv["page_type"] == "checkout" for pv in s["page_views"]) for s in sessions))
print("Avg page views per session:", round(sum(len(s["page_views"]) for s in sessions) / len(sessions), 1))

Page-view mix:


,page_views
product_detail,691101
category_listing,398658
search,287089
home,253700
cart,194526
checkout,102686
confirmation,70818


Sessions with a non-empty cart: 130170
Sessions that visited the checkout page: 87087
Avg page views per session: 10.0


###  Briefly we realize the following 
* `data/raw/` contains `users.json`, `products.json`, `categories.json`, `transactions.json`, `sessions_0.json` …
* `outputs/tables/data_quality_findings.csv` exists.
* You understand the 4 key facts that shape the design: **no subcategory_id**, **no age**, **`conversion_status` ≠ real purchases**, **~half of transactions have no session**.

